# FlexSDS Server on Google Colab (vLLM Speculative Decoding Engine)

This notebook launches the FlexSDS speculative decoding serving engine with EAGLE-3.1 / Adaptive Draft on Google Colab (T4 / L4 / A100 GPU) and establishes a public Cloudflare Tunnel (or VS Code Remote Tunnel) to connect with the `flexsds` CLI.

In [ ]:
# STEP 1: SYNC SOURCE CODE AND INSTALL ENVIRONMENT
import os
import subprocess

REPO_URL = "https://github.com/theflyingkhui04/do-an-ptit-d22.git"
TARGET_DIR = "/content/do-an-ptit-d22"

if not os.path.exists(TARGET_DIR):
    print("==> Fetching FlexSDS source repository...")
    subprocess.run(["git", "clone", REPO_URL, TARGET_DIR], check=True)
else:
    print("==> Pulling latest repository updates...")
    subprocess.run(["git", "-C", TARGET_DIR, "pull"], check=True)

print("==> Installing vLLM v0.30.0 and required dependencies...")
subprocess.run(["pip", "install", "-q", "-r", f"{TARGET_DIR}/requirements-server.txt"], check=True)
subprocess.run(["pip", "uninstall", "-y", "torchaudio"], check=False)
print("✔ Environment setup complete!")

In [ ]:
# STEP 2: INSTALL CLOUDFLARE TUNNEL (CLOUDFLARED)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null
print("✔ Cloudflared is ready!")

In [ ]:
#@title STEP 3: CONFIGURE & LAUNCH vLLM SERVING { display-mode: "form" }
import json
import os
import re
import subprocess
import time
import urllib.error
import urllib.request

# Configuration Parameters (synchronized by FlexSDS CLI or editable below)
TARGET_MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"  #@param {type:"string"}
DRAFT_MODEL = "duckhoi/qwen2.5-7b-eagle3.1-speculator"  #@param {type:"string"}
DRAFT_METHOD = "eagle3"  #@param ["eagle", "eagle3", "draft_model"]
QUANTIZATION = "awq"  #@param ["awq", "gptq", "none"]
GPU_MEMORY_UTILIZATION = 0.85  #@param {type:"number"}
MAX_MODEL_LEN = 4096  #@param {type:"integer"}
MAX_NUM_SEQS = 16  #@param {type:"integer"}
NUM_SPECULATIVE_TOKENS = 4  #@param {type:"integer"}
ENABLE_ADAPTIVE = True  #@param {type:"boolean"}
HF_TOKEN = ""  #@param {type:"string"}

# Terminate any lingering background vLLM or cloudflared instances from previous attempts
subprocess.run(["pkill", "-9", "-f", "vllm"], check=False)
subprocess.run(["pkill", "-9", "-f", "cloudflared"], check=False)
time.sleep(2)

TARGET_DIR = TARGET_DIR if "TARGET_DIR" in globals() else os.path.abspath("/content/do-an-ptit-d22")
os.environ["PYTHONPATH"] = f"{TARGET_DIR}:{os.environ.get('PYTHONPATH', '')}"
os.environ["ADAPTIVE_ENABLE_VRAM_GUARD"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN.strip()

# Build speculative decoding config
spec_config = {
    "method": DRAFT_METHOD,
    "model": DRAFT_MODEL,
    "num_speculative_tokens": NUM_SPECULATIVE_TOKENS,
    "draft_sample_method": "greedy",
}

server_cmd = [
    "vllm", "serve", TARGET_MODEL,
    "--host", "0.0.0.0",
    "--port", "8000",
    "--served-model-name", "qwen2.5-7b-adaptive",
    "--dtype", "half" if QUANTIZATION != "none" else "bfloat16",
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
    "--max-model-len", str(MAX_MODEL_LEN),
    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--generation-config", "vllm",
    "--speculative-config", json.dumps(spec_config),
    "--per-request-spec-decode-metrics", "summary",
]

if QUANTIZATION != "none":
    server_cmd.extend(["--quantization", QUANTIZATION])

# Adaptive AsyncScheduler applies to 'draft_model' speculative decoding.
# Native EAGLE-3.1 engine is self-contained and operates without custom scheduler hooks.
if ENABLE_ADAPTIVE and DRAFT_METHOD == "draft_model":
    spec_config["use_heterogeneous_vocab"] = True
    spec_config["num_speculative_tokens_per_batch_size"] = [[1, 32, NUM_SPECULATIVE_TOKENS]]
    server_cmd[server_cmd.index("--speculative-config") + 1] = json.dumps(spec_config)
    server_cmd.extend([
        "--scheduler-cls", "src.server.adaptive_scheduler.AdaptiveAsyncScheduler",
        "--distributed-executor-backend", "src.server.lineage_executor.LineageUniProcExecutor",
    ])
    print("ℹ AdaptiveAsyncScheduler enabled for draft_model.")
elif DRAFT_METHOD == "eagle3":
    print("ℹ Native EAGLE-3.1 engine active.")

# Launch vLLM process and stream startup logs
log_file = "/content/vllm_server.log"
with open(log_file, "w") as f:
    f.write("=== Starting vLLM Server ===\n")

print(f"\n==> Launching vLLM serving for: {TARGET_MODEL}")
print(f"    Draft Model: {DRAFT_MODEL} (method: {DRAFT_METHOD})")
print(f"    Log destination: {log_file}")

log_out = open(log_file, "a")
vllm_proc = subprocess.Popen(
    server_cmd,
    stdout=log_out,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

print("==> Monitoring startup progress (downloading weights & allocating GPU VRAM)...")
start_time = time.time()
last_activity_time = time.time()
server_ready = False
log_reader = open(log_file, "r")
log_reader.seek(0)
MAX_TOTAL_WAIT = 900  # 15 minutes timeout for download & graph warmup

while time.time() - start_time < MAX_TOTAL_WAIT:
    if vllm_proc.poll() is not None:
        print(f"\n❌ vLLM process terminated with exit code: {vllm_proc.poll()}")
        log_reader.seek(0)
        lines = log_reader.readlines()
        print("----- Last 30 lines of vLLM error log: -----")
        print("".join(lines[-30:]))
        raise RuntimeError("vLLM failed to start. Review the log output above.")

    new_lines = log_reader.readlines()
    if new_lines:
        last_activity_time = time.time()
        for line in new_lines:
            s = line.strip()
            if s:
                print(f"  [vLLM] {s}")

    try:
        req = urllib.request.Request("http://127.0.0.1:8000/health", headers={"User-Agent": "FlexSDS-Probe"})
        with urllib.request.urlopen(req, timeout=1) as resp:
            if resp.status == 200:
                server_ready = True
                break
    except Exception:
        pass

    time.sleep(3)

if not server_ready:
    raise TimeoutError(f"vLLM server startup timed out after {MAX_TOTAL_WAIT // 60} minutes.")

print("\n✔ vLLM is healthy and serving requests!")
print("==> Activating Cloudflare public tunnel...")

tunnel_log_file = "/content/cloudflared.log"
tunnel_out = open(tunnel_log_file, "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=tunnel_out,
    stderr=subprocess.STDOUT,
)

public_url = None
url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
tunnel_reader = open(tunnel_log_file, "r")

for _ in range(40):
    if tunnel_proc.poll() is not None:
        print("❌ Cloudflared process exited unexpectedly.")
        break
    time.sleep(1)
    tunnel_reader.seek(0)
    text = tunnel_reader.read()
    match = url_pattern.search(text)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n" + "="*70)
    print("🚀 FLEXSDS SERVING ENGINE IS READY ON GOOGLE COLAB!")
    print(f"👉 Public Tunnel URL: {public_url}")
    print(f"👉 Serving Endpoint:  {public_url}/v1")
    print(f"👉 Start Chatting:    flexsds chat --endpoint {public_url}/v1")
    print("="*70 + "\n")
else:
    print("⚠️ Could not detect tunnel URL automatically. Check cloudflared.log.")

try:
    while True:
        if vllm_proc.poll() is not None:
            print("❌ vLLM process stopped.")
            break
        if tunnel_proc.poll() is not None:
            print("❌ Tunnel process stopped.")
            break
        time.sleep(5)
except KeyboardInterrupt:
    print("\nTerminating server processes...")
    vllm_proc.terminate()
    tunnel_proc.terminate()


## Optional: VS Code Remote Tunnel

If you prefer to connect directly to the Google Colab instance via VS Code Desktop using a remote tunnel (similar to `logs/Khoi/M1-01/vscode_cli.ipynb`), run the cell below.

In [ ]:
#@title STEP 4 (OPTIONAL): LAUNCH VIA VS CODE REMOTE TUNNEL { display-mode: "form" }
import os

TUNNEL_NAME = "colab-t4"  #@param {type:"string"}

if not os.path.exists("./code"):
    print("--> Downloading Visual Studio Code CLI...")
    !wget -q -O vscode_cli.tar.gz "https://update.code.visualstudio.com/latest/cli-alpine-x64/stable" && tar -xf vscode_cli.tar.gz

print(f"--> Launching VS Code Remote Tunnel with name '{TUNNEL_NAME}'...")
!./code tunnel --accept-server-license-terms --name {TUNNEL_NAME}
